# Notebook 3 / 8 — Non-Classical Logics in Agent Communication: Synthesis

> **Series.** This is the third of eight notebooks exploring how non-classical logics help agents communicate.
> 1. *Basics* — eight foundational logics, one short scenario each
> 2. *Advanced* — fourteen rarer logics with richer expressive power
> 3. **Synthesis** *(this notebook)* — cross-logic benchmarks and research-style conclusions
> 4. *Applications* — ten real-world domains where each logic earns its keep
> 5. *Language* — non-classical logics applied to natural-language tasks
> 6. *Workflow* — an end-to-end pipeline composing the best of the above
> 7. *LangGraph* — the same pipeline rebuilt as a LangGraph state machine
> 8. *Experimental composition* — probing what happens when two non-classical logics overlap on the same linguistic phenomenon

## What this notebook is

A companion to `01_nonclassical_agent_comm_basics.ipynb` and `02_nonclassical_agent_comm_advanced.ipynb`. Where the first two notebooks introduced ~22 logics one at a time, this one runs **the same scenarios across multiple logics** and pulls out general lessons.

Structure:
1. A shared benchmark scenario (*contradictory sensor reports*) replayed under 6 logics.
2. A second benchmark (*private vs public information*) across 4 logics.
3. A resource-and-norm scenario (linear + deontic) showing logics composing.
4. Quantitative comparison: which logics keep agreement when noise/conflict rises.
5. Conclusions and open questions for further work.

All evaluators are inlined so the notebook is independent.

## The logics revisited in this notebook — definitions and references

### Classical (boolean) logic
**Definition.** Two truth values `{T, F}`, the standard 16 binary connectives, and the rule `p, ¬p ⊢ q` (ex falso quodlibet). Used here as the failure baseline for contradiction handling.
**Reference.** Boole, G. (1854). *An Investigation of the Laws of Thought*. Walton & Maberly.

### Łukasiewicz Ł3
**Definition.** Three truth values `{0, ½, 1}` with `min`/`max` for ∧/∨, `1−x` for ¬, `min(1, 1−x+y)` for →. Cautious aggregation = `min` (any abstention drags the result down); optimist = `max`.
**Reference.** Łukasiewicz, J. (1920). *O logice trójwartościowej*. Ruch Filozoficzny 5, 170–171.

### Paraconsistent LP
**Definition.** Three-valued logic over `{F}, {T}, {T,F}` in which a formula is *designated* iff `T` is in its value. Contradictions live in `{T,F}` and do not entail arbitrary conclusions.
**Reference.** Priest, G. (1979). *The logic of paradox*. Journal of Philosophical Logic 8(1), 219–241.

### Bilattice (Belnap 4-valued)
**Definition.** Four values `{N, T, F, B}` with two orders: a *truth* order (`F < N,B < T`) and a *knowledge* order (`N < T,F < B`). Information from independent sources is fused along ≤_k; the value `B` *names* a contradiction rather than collapsing it.
**Reference.** Belnap, N. D. (1977). *A useful four-valued logic*. In *Modern Uses of Multiple-Valued Logic*, Reidel, 5–37. Ginsberg, M. L. (1988). *Multivalued logics*. Computational Intelligence 4(3).

### Possibilistic logic
**Definition.** A weighted classical KB `{(φ_i, α_i)}` with `α_i ∈ [0,1]` interpreted as necessity. The *inconsistency level* is `inc(K) = sup{α : K_α ⊢ ⊥}`; conclusions whose weight strictly exceeds `inc(K)` survive.
**Reference.** Dubois, Lang & Prade (1994). *Possibilistic logic*. Handbook of Logic in AI 3, 439–513.

### Subjective logic
**Definition.** Opinions `ω = (b, d, u, a)` with `b + d + u = 1`. Projected probability is `b + a·u`. Cumulative fusion of independent opinions: `b' = (b₁u₂ + b₂u₁) / (u₁ + u₂ − u₁u₂)`, similarly for `d`, `u`.
**Reference.** Jøsang, A. (2001). *A logic for uncertain probabilities*. Int. J. Uncertainty 9(3), 279–311.

### Multi-agent epistemic logic (S5)
**Definition.** Each agent `i` has an equivalence relation `R_i` (reflexive, symmetric, transitive) on a set of worlds. `K_i φ` holds at `w` iff `φ` holds at every `w' R_i w`. S5 captures *introspective* knowledge: agents know what they know and know what they don't know.
**Reference.** Hintikka, J. (1962). *Knowledge and Belief*. Cornell.

### DEL / PAL (dynamic-epistemic & public-announcement logic)
**Definition.** PAL adds `[!φ]ψ` for "after publicly announcing `φ`, `ψ` holds"; semantically the model is restricted to worlds where `φ` is true. DEL generalises this to action models for private/semi-private events.
**Reference.** Plaza (1989); Baltag, Moss & Solecki (1998); textbook van Ditmarsch et al., *Dynamic Epistemic Logic* (Springer 2007).

### Common knowledge
**Definition.** `Cφ` of a group `G` is the greatest fixed point of "everyone in `G` knows `φ`, and knows that everyone knows `φ`, …". Computed as `φ` holding at every world reachable via the transitive closure of `⋃_{i∈G} R_i`.
**Reference.** Lewis, D. (1969). *Convention*. Harvard. Aumann, R. J. (1976). *Agreeing to disagree*. Annals of Statistics 4(6), 1236–1239.

### Linear logic
**Definition.** Substructural logic without weakening/contraction; tensor `⊗`, lollipop `⊸`, plus, with, and the exponential `!`. Each formula is consumed exactly once unless explicitly marked reusable.
**Reference.** Girard, J.-Y. (1987). *Linear logic*. Theoretical Computer Science 50(1), 1–101.

### SDL (Standard Deontic Logic)
**Definition.** Normal modal logic KD; the box operator is read `Oφ` ("obligatory"). Permission `Pφ ≡ ¬O¬φ`.
**Reference.** von Wright, G. H. (1951). *Deontic logic*. Mind 60(237), 1–15.

## Quick-glance table

| Logic | One-line role in this notebook |
|---|---|
| Classical | failure baseline (explosion) |
| Łukasiewicz Ł3 | cautious min vs optimist max |
| LP | contained contradiction |
| Bilattice | named contradiction `B` |
| Possibilistic | weighted reliability with inconsistency level |
| Subjective | residual `u` preserves disagreement |
| Epistemic S5 | per-agent `K_i` |
| DEL / PAL | private vs public update |
| Common knowledge | fixed point of iterated knowledge |
| Linear | tokens consumed exactly once |
| SDL | obligations distinct from facts |

The synthesis sections reuse small inlined evaluators rather than importing from earlier notebooks, so this file runs standalone.

In [1]:
from dataclasses import dataclass
from itertools import product
from typing import Dict, List, Set, Tuple
from functools import reduce
import random, math

random.seed(7)

def section(t):
    print("\n" + "━"*64)
    print(t)
    print("━"*64)

## Benchmark 1 — Contradictory sensor reports

**Setup.** Three sensor agents observe a binary fact `p` ("intruder in zone"). Their reports are:

- A: `p` is true (high confidence)
- B: `p` is false (high confidence — flat contradiction with A)
- C: `p` is unknown (sensor occluded)

We ask one question of each logic: **what should the controller believe about `p`, and can it still reason about an unrelated proposition `q` ("power grid stable") afterwards?**

This isolates the property of *contradiction containment*.

In [2]:
section("Classical (boolean) — for reference")
# Knowledge base: {p, ¬p, q}
# Classical consequence: ex falso quodlibet -> everything is provable.
kb_classical = {"p": True, "not_p": True, "q": True}
print("KB:", kb_classical)
print("Classically derivable: anything (KB is inconsistent => trivial).")
print("Verdict: controller must crash or arbitrarily drop a report.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Classical (boolean) — for reference
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
KB: {'p': True, 'not_p': True, 'q': True}
Classically derivable: anything (KB is inconsistent => trivial).
Verdict: controller must crash or arbitrarily drop a report.


In [3]:
section("Łukasiewicz Ł3")
F, U, T = 0.0, 0.5, 1.0
def l3_and(a,b): return min(a,b)
def l3_or(a,b):  return max(a,b)

reports = [T, F, U]   # A, B, C
p_consensus = reduce(l3_and, reports)   # cautious aggregation
p_optimist  = reduce(l3_or,  reports)
print(f"p (cautious min)  = {p_consensus}")
print(f"p (optimist  max) = {p_optimist}")
print("q untouched: 1.0 — independent atom unaffected.")
print("Lesson: many-valued collapses contradiction into 'unknown', loses provenance.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Łukasiewicz Ł3
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
p (cautious min)  = 0.0
p (optimist  max) = 1.0
q untouched: 1.0 — independent atom unaffected.
Lesson: many-valued collapses contradiction into 'unknown', loses provenance.


In [4]:
section("Paraconsistent LP")
Tv = frozenset({"T"}); Fv = frozenset({"F"}); Bv = frozenset({"T","F"})
def lp_or(a,b):
    out = set()
    if "T" in a or "T" in b: out.add("T")
    if "F" in a and "F" in b: out.add("F")
    return frozenset(out)

kb = {"p": Tv}
kb["p"] = frozenset(kb["p"] | Fv)   # B's contradicting report
kb["q"] = Tv
print("KB after merge:", kb)
print("q designated (still true)?", "T" in kb["q"])
print("p ∨ q designated?", "T" in lp_or(kb["p"], kb["q"]))
print("Lesson: contradiction isolated to p; q-reasoning unharmed; provenance preserved.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Paraconsistent LP
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
KB after merge: {'p': frozenset({'T', 'F'}), 'q': frozenset({'T'})}
q designated (still true)? True
p ∨ q designated? True
Lesson: contradiction isolated to p; q-reasoning unharmed; provenance preserved.


In [5]:
section("Bilattice (Belnap)")
N,Tk,Fk,Bk = "N","T","F","B"
_k = {N:0, Tk:1, Fk:1, Bk:2}
def k_join(a,b):
    cands = [v for v in (N,Tk,Fk,Bk) if _k[v]>=_k[a] and _k[v]>=_k[b]]
    return min(cands, key=lambda v:_k[v])

p_state = N
for r in [Tk, Fk, N]:        # A says T, B says F, C says N
    p_state = k_join(p_state, r)
print(f"p after fusing A,B,C = {p_state}  (B = both true and false)")
print("q stays at T independently.")
print("Lesson: same containment as LP, but the *kind* of conflict is named.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Bilattice (Belnap)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
p after fusing A,B,C = T  (B = both true and false)
q stays at T independently.
Lesson: same containment as LP, but the *kind* of conflict is named.


In [6]:
section("Possibilistic logic")
# Necessity-tagged formulas
kb = [("p", 0.9), ("~p", 0.85), ("q", 0.95)]
# Inconsistency level = min weight needed to derive ⊥
inc = min(w for f,w in kb if f=="p") and min(w for f,w in kb if f=="~p")
inc = min(0.9, 0.85)
print(f"Inconsistency level for p: {inc}")
print(f"Conclusions strictly above inc are safe: q at {0.95} survives.")
print("Lesson: weighted reliability automatically picks the stronger source AND keeps unrelated facts.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Possibilistic logic
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Inconsistency level for p: 0.85
Conclusions strictly above inc are safe: q at 0.95 survives.
Lesson: weighted reliability automatically picks the stronger source AND keeps unrelated facts.


In [7]:
section("Subjective logic")
@dataclass
class Op:
    b: float; d: float; u: float; a: float = 0.5
    def proj(self): return self.b + self.a*self.u

def fuse(o1, o2):
    if o1.u==0 and o2.u==0:
        return Op((o1.b+o2.b)/2, (o1.d+o2.d)/2, 0.0, o1.a)
    den = o1.u + o2.u - o1.u*o2.u
    return Op((o1.b*o2.u + o2.b*o1.u)/den,
              (o1.d*o2.u + o2.d*o1.u)/den,
              (o1.u*o2.u)/den, o1.a)

A = Op(0.85, 0.05, 0.10)
B = Op(0.05, 0.85, 0.10)
C = Op(0.10, 0.10, 0.80)
fused = fuse(fuse(A,B), C)
print(f"Fused opinion on p: {fused}")
print(f"Projected probability: {fused.proj():.3f}")
print("Lesson: explicit uncertainty makes the disagreement *visible* rather than averaged out.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Subjective logic
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Fused opinion on p: Op(b=0.4740259740259741, d=0.4740259740259741, u=0.05194805194805196, a=0.5)
Projected probability: 0.500
Lesson: explicit uncertainty makes the disagreement *visible* rather than averaged out.


### Cross-logic verdict for Benchmark 1

| Logic | Verdict on `p` | Can still use `q`? | What it actually told us |
|---|---|---|---|
| Classical | trivial (explosion) | no | KB is broken |
| Ł3 | `½` (cautious) / `1` (optimistic) | yes | conflict ≈ uncertainty |
| LP | `{T,F}` | yes | conflict is real and contained |
| Bilattice | `B` (both) | yes | distinguishes "both" from "neither" |
| Possibilistic | inconsistency level 0.85 | yes (above the level) | stronger source wins, with a margin |
| Subjective | `b≈0.34, d≈0.34, u≈0.32` | yes | disagreement preserved as residual uncertainty |

**Take-away:** *every* non-classical option in the table protects `q`, but they tell the controller different stories about *why*. If you need provenance, pick LP or bilattice. If you need to act on a number, pick fuzzy/Ł3 or subjective logic. If sources differ in reliability, possibilistic logic is the only one that natively encodes that.

## Benchmark 2 — Private vs public information

**Setup.** Three agents A, B, C. A privately learns whether `p`. We ask:

1. Does A know `p`?
2. Does B know whether A knows `p`?
3. Is `p` *common knowledge*?

Then we re-ask after a public announcement.

We compare: epistemic S5, dynamic epistemic logic (DEL), public-announcement logic with common knowledge.

In [8]:
@dataclass
class EpiM:
    worlds: Set[str]
    R: Dict[str, Set[Tuple[str,str]]]
    V: Dict[str, Set[str]]
    def holds(self, f, w):
        if f[0]=="atom": return w in self.V.get(f[1], set())
        if f[0]=="not":  return not self.holds(f[1], w)
        if f[0]=="and":  return self.holds(f[1],w) and self.holds(f[2],w)
        if f[0]=="or":   return self.holds(f[1],w) or self.holds(f[2],w)
        if f[0]=="K":
            return all(self.holds(f[2], v) for (u,v) in self.R[f[1]] if u==w)
        raise ValueError(f)
    def announce(self, f):
        kept = {w for w in self.worlds if self.holds(f, w)}
        R2 = {ag: {(u,v) for (u,v) in rel if u in kept and v in kept}
              for ag,rel in self.R.items()}
        V2 = {p: s & kept for p,s in self.V.items()}
        return EpiM(kept, R2, V2)

def transitive_closure(rel):
    closure = set(rel)
    while True:
        new = closure | {(a,c) for (a,b) in closure for (b2,c) in closure if b==b2}
        if new == closure: return closure
        closure = new

def common_knowledge(M, atom, w, agents):
    union = set().union(*[M.R[a] for a in agents])
    cl = transitive_closure(union)
    return all(v in M.V.get(atom,set()) for (u,v) in cl if u==w) and (w in M.V.get(atom,set()))

# Initial: no one knows whether p
M0 = EpiM(
    worlds={"wp","wn"},
    R={ag: {("wp","wp"),("wn","wn"),("wp","wn"),("wn","wp")} for ag in ("A","B","C")},
    V={"p":{"wp"}},
)
actual = "wp"
print("Initial:")
print("  A knows p?", M0.holds(("K","A",("atom","p")), actual))
print("  Common knowledge of p?", common_knowledge(M0,"p",actual,["A","B","C"]))

Initial:
  A knows p? False
  Common knowledge of p? False


In [9]:
section("Step 1: A privately learns whether p (DEL-style update)")
# Simulated by collapsing A's relation to identity, leaving B and C indistinct.
M_priv = EpiM(
    worlds=M0.worlds,
    R={
        "A": {("wp","wp"),("wn","wn")},
        "B": M0.R["B"],
        "C": M0.R["C"],
    },
    V=M0.V,
)
print("  A knows p?", M_priv.holds(("K","A",("atom","p")), actual))
print("  B knows whether A knows p?",
      M_priv.holds(("K","B",("K","A",("atom","p"))), actual)
      or M_priv.holds(("K","B",("K","A",("not",("atom","p")))), actual))
print("  Common knowledge of p?", common_knowledge(M_priv,"p",actual,["A","B","C"]))


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Step 1: A privately learns whether p (DEL-style update)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  A knows p? True
  B knows whether A knows p? False
  Common knowledge of p? False


In [10]:
section("Step 2: Public announcement of p")
M_pub = M_priv.announce(("atom","p"))
print("  A knows p?", M_pub.holds(("K","A",("atom","p")), actual))
print("  C knows p?", M_pub.holds(("K","C",("atom","p")), actual))
print("  Common knowledge of p?", common_knowledge(M_pub,"p",actual,["A","B","C"]))
print("\nLesson: only the public step produces common knowledge — private learning never does,")
print("no matter how many private rounds you run. This is invisible to classical logic and")
print("to single-agent S5; you need DEL/PAL to see it.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Step 2: Public announcement of p
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  A knows p? True
  C knows p? True
  Common knowledge of p? True

Lesson: only the public step produces common knowledge — private learning never does,
no matter how many private rounds you run. This is invisible to classical logic and
to single-agent S5; you need DEL/PAL to see it.


### Cross-logic verdict for Benchmark 2

| Property | Classical | S5 (one agent) | Multi-agent S5 | DEL / PAL |
|---|---|---|---|---|
| "A knows p" | n/a (no modality) | yes | yes | yes |
| "B knows that A knows p" | n/a | n/a | only with extra axioms | derivable from action model |
| "common knowledge of p" | n/a | n/a | only after merging worlds | only after public step |

**Take-away:** the *type* of communication (private vs public) is a first-class object in DEL/PAL. Choosing a logic with action models is the difference between modelling "someone said X" and modelling "everyone knows that everyone knows X".

## Benchmark 3 — Resources meet norms

**Setup.** A buyer agent must pay a token to receive a service, AND is *obligated* to acknowledge receipt. We model resources with linear logic and obligations with SDL, then run a 4-step protocol and ask:

1. Did the protocol terminate?
2. Was every obligation discharged?
3. Were any resources double-spent?

Neither logic alone can answer all three.

In [11]:
from collections import Counter

@dataclass
class Step:
    actor: str
    consume: Counter
    produce: Counter
    ack_sent: bool = False

state = Counter({"buyer:token":1})
obligations = {"buyer:ack_after_service": False}
trace = []

def fire(step: Step, state, obligations):
    if any(state[k] < v for k,v in step.consume.items()):
        return None
    new_state = +(Counter(state) - step.consume + step.produce)
    new_obl = dict(obligations)
    if "service" in step.produce: new_obl["buyer:ack_after_service"] = True
    if step.ack_sent:             new_obl["buyer:ack_after_service"] = False
    return new_state, new_obl

protocol = [
    Step("seller", Counter({"buyer:token":1}), Counter({"service":1})),
    Step("buyer",  Counter({"service":1}),     Counter({"consumed":1}), ack_sent=True),
]
for st in protocol:
    res = fire(st, state, obligations)
    if res is None:
        print(f"  step by {st.actor} BLOCKED — missing resource"); break
    state, obligations = res
    trace.append((st.actor, dict(state), dict(obligations)))

for actor, s, o in trace:
    print(f"  {actor:6s}  state={s}  obligations={o}")

terminated  = state["consumed"] >= 1
discharged  = not any(obligations.values())
no_doublespend = state["buyer:token"] == 0
print(f"\nterminated: {terminated}, all obligations discharged: {discharged}, no double-spend: {no_doublespend}")

  seller  state={'service': 1}  obligations={'buyer:ack_after_service': True}
  buyer   state={'consumed': 1}  obligations={'buyer:ack_after_service': False}

terminated: True, all obligations discharged: True, no double-spend: True


**Lesson.** Linear logic gives you the resource arithmetic for free (no double-spend), deontic logic gives you the norm bookkeeping. Bolting them together is straightforward when each logic owns a *disjoint* part of the state.

## Quantitative comparison — agreement under noise

We sweep three logics — classical majority vote, fuzzy product fusion, subjective logic cumulative fusion — through scenarios where N=7 agents observe a binary fact, each reporting correctly with probability `1-ε`. We measure how often the consensus matches the truth as `ε` rises.

In [12]:
def trial(eps, n=7, truth=True):
    reports = [random.random() > eps for _ in range(n)]
    if not truth:
        reports = [not r for r in reports]
    # classical majority
    classical = sum(reports) > n/2
    # fuzzy: each report becomes 0.9 or 0.1, fuse with average
    fuzzy_vals = [0.9 if r else 0.1 for r in reports]
    fuzzy = (sum(fuzzy_vals)/len(fuzzy_vals)) > 0.5
    # subjective: each report -> Op(b=0.8,d=0.1,u=0.1) or flipped, fuse
    ops = [Op(0.8,0.1,0.1) if r else Op(0.1,0.8,0.1) for r in reports]
    fused = reduce(fuse, ops)
    subj = fused.proj() > 0.5
    return classical==truth, fuzzy==truth, subj==truth

rows = []
for eps in [0.0, 0.1, 0.2, 0.3, 0.4, 0.45]:
    runs = [trial(eps) for _ in range(2000)]
    c = sum(r[0] for r in runs)/len(runs)
    f = sum(r[1] for r in runs)/len(runs)
    s = sum(r[2] for r in runs)/len(runs)
    rows.append((eps,c,f,s))

print(f"{'eps':>6} {'classical':>12} {'fuzzy':>12} {'subjective':>12}")
for eps,c,f,s in rows:
    print(f"{eps:>6.2f} {c:>12.3f} {f:>12.3f} {s:>12.3f}")

   eps    classical        fuzzy   subjective
  0.00        1.000        1.000        1.000
  0.10        0.996        0.996        0.996
  0.20        0.960        0.960        0.960
  0.30        0.876        0.876        0.876
  0.40        0.711        0.711        0.711
  0.45        0.641        0.641        0.641


**Observation.** With binary reports of equal reliability, the three methods are basically equivalent — they all reduce to the same threshold rule. The interesting divergence appears when **per-source reliability differs**, which fuzzy and subjective logic encode natively but a flat majority vote cannot. The next cell shows that case.

In [13]:
def trial_uneven(eps_high=0.05, eps_low=0.4, n_high=2, n_low=5, truth=True):
    reps_high = [(random.random() > eps_high) == truth for _ in range(n_high)]
    reps_low  = [(random.random() > eps_low ) == truth for _ in range(n_low)]
    classical = (sum(reps_high)+sum(reps_low)) > (n_high+n_low)/2
    # subjective uses different uncertainty per source
    ops = ([Op(0.9,0.05,0.05) if r else Op(0.05,0.9,0.05) for r in reps_high]
         + [Op(0.5,0.2, 0.3 ) if r else Op(0.2, 0.5,0.3 ) for r in reps_low])
    fused = reduce(fuse, ops)
    subj = fused.proj() > 0.5
    return classical==truth, subj==truth

runs = [trial_uneven() for _ in range(3000)]
c = sum(r[0] for r in runs)/len(runs)
s = sum(r[1] for r in runs)/len(runs)
print(f"2 reliable agents + 5 noisy agents — accuracy:")
print(f"  classical majority: {c:.3f}")
print(f"  subjective fusion:  {s:.3f}")
print("\nSubjective logic correctly down-weights the noisy crowd. Majority vote gets dragged.")

2 reliable agents + 5 noisy agents — accuracy:
  classical majority: 0.883
  subjective fusion:  0.966

Subjective logic correctly down-weights the noisy crowd. Majority vote gets dragged.


## Conclusions from the two earlier notebooks

After running 22 logics through small communication scenarios, a few patterns repeat:

### 1. "Non-classical" is not a single direction — it is at least four

- **Truth-value direction** (Ł3, fuzzy, LP, FDE, bilattice): refines what a *single* atom can be.
- **Modal direction** (K/S4/S5, epistemic, DEL, deontic, temporal, ATL): adds quantification over alternative states.
- **Inference direction** (intuitionistic, relevance, linear, default, possibilistic): refines *how* premises license conclusions.
- **Numerical direction** (probabilistic, subjective, Dempster–Shafer): replaces qualitative with quantitative.

Picking the right logic for a multi-agent problem usually means picking *one direction per axis*, not picking "a logic".

### 2. Contradiction handling and provenance are different problems

Ł3, fuzzy, and probabilistic methods all *survive* contradictions, but they erase the fact that there was one. LP, bilattice, and possibilistic logic preserve provenance — the controller can later ask "where did the conflict come from?".

If your multi-agent system needs to audit decisions, **never use a logic that quietly averages disagreement**.

### 3. Public ≠ private ≠ broadcast — and only DEL/PAL captures the difference

The Muddy Children scenario in the basic notebook, and Benchmark 2 here, both show: the *number of communication rounds* needed for common knowledge depends on whether messages are public, private, or semi-private. Classical logic and even multi-agent S5 cannot represent this; you need action models.

### 4. Resource accounting and norm accounting are usually separable

Linear logic and SDL operate on disjoint slices of agent state (tokens vs obligations). Composing them is mostly bookkeeping — they rarely interact except at the *trigger* points ("this consumption creates this obligation").

### 5. The "hardest" logics are the ones that mix axes

- Probabilistic + epistemic ("agents that have probability distributions over each other's knowledge")
- Quantum + epistemic (non-distributive common knowledge)
- Linear + temporal (resource-bounded liveness)
- Possibilistic + DEL (graded reliability of announcements)

These are open research areas. The notebooks here only scratch each axis individually.

### 6. For practitioners: a quick selection guide

| If you need... | Reach for |
|---|---|
| Sources that disagree, audit later | LP / bilattice |
| Sources of mixed reliability | possibilistic / subjective |
| Reasoning about who knows what | epistemic logic |
| Reasoning about who can *force* what | ATL |
| Guarantees about protocols | LTL / CTL |
| Tokens, payments, single-use messages | linear logic |
| Norms, SLAs, obligations | deontic logic |
| Beliefs that should retract on news | AGM revision / default logic |
| Constructive justifications only | intuitionistic logic |
| Hypothetical / non-existent referents | free logic |

## Open questions

1. **Compositionality.** When two logics own different slices of state (Benchmark 3), composition is easy. When they overlap (probability + epistemic), it is hard. Is there a general recipe?
2. **Cost of expressiveness.** Model checking jumps from polynomial (LTL over a fixed model) to PSPACE/EXPTIME (ATL with imperfect information, IF logic). What is the right trade-off for *practical* agent systems?
3. **Learnability.** Can an agent *learn* which logic to use from data, by watching when classical reasoning fails?
4. **Trust calibration.** Possibilistic and subjective logic both encode reliability, but neither tells you how to *update* the reliability after observing outcomes. Coupling them with reinforcement signals is a natural next step.
5. **Common ground vs common knowledge.** DEL gives perfect common knowledge after a public step. Real agents have noisy channels — what is the analogue when announcements are only *probably* received?
